In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM songs ORDER BY plays DESC", conn)
    print(df)

In [ ]:
query = """
WITH genre_stats AS (
    SELECT 
        artists.genre,
        songs.title,
        songs.plays,
        SUM(songs.plays) OVER (PARTITION BY artists.genre) as genre_total_plays,
        RANK() OVER (PARTITION BY artists.genre ORDER BY songs.plays DESC) as rank_in_genre
    FROM songs
    JOIN artists ON songs.artist = artists.name
)
SELECT *
FROM genre_stats
WHERE rank_in_genre = 1
ORDER BY genre_total_plays DESC
"""

with engine.connect() as conn:
    genre_df = pd.read_sql(query, conn)

print(genre_df)

In [ ]:
# Load all three tables into pandas and do analysis in Python
with engine.connect() as conn:
    songs_df = pd.read_sql("SELECT * FROM songs", conn)
    artists_df = pd.read_sql("SELECT * FROM artists", conn)

# Merge in pandas - same as SQL JOIN
merged = songs_df.merge(artists_df, left_on="artist", right_on="name")
# left_table.merge(right_table, left_on="column_in_left", right_on="column_in_right")

# Group be genre
genre_summary = merged.groupby("genre")["plays"].agg(
    total_plays="sum", # create total_plays, apply sum
    avg_plays="mean", # create avg_plays, apply mean
    song_count="count" # create song_count, apply count
).sort_values("total_plays", ascending=False)

print(genre_summary)

In [ ]:
with engine.connect() as conn:
    songs_df = pd.read_sql("SELECT * FROM songs", conn)
    artists_df = pd.read_sql("SELECT * FROM artists", conn)
    albums_df = pd.read_sql("SELECT * FROM albums", conn)

# Rename before merging to avoid confusion
artists_df = artists_df.rename(columns={
    "id": "artist_id",
    "name": "artist_name"
})

# Step 1 - songs + artists, print columns to verify
songs_artists = songs_df.merge(
    artists_df,
    left_on="artist",
    right_on="artist_name",
    how="left"
)
print("After first merge:")
print(songs_artists.columns.tolist())

# Step 2 - add albums
full_df = songs_artists.merge(
    albums_df,
    left_on="artist_id",
    right_on="artist_id",
    how="left"
)
print("\nAfter second merge:")
print(full_df.columns.tolist())

# Select and rename meaningful columns
result = full_df[[
    "title_x", "plays", "genre", 
    "country", "title_y", "release_year"
]]
result.columns = ["song", "plays", "genre", "country", "album", "album_year"]

# Check for missing values
print("\nMissing values per column:")
print(result.isnull().sum())

print("\nFull result:")
print(result.sort_values("plays", ascending=False).to_string())

In [ ]:
with engine.connect() as conn:
    query = """
        SELECT 
            s.title as song,
            s.artist,
            s.plays
        FROM songs s
        LEFT JOIN artists ar ON s.artist = ar.name
        LEFT JOIN albums al ON ar.id = al.artist_id
        WHERE al.id IS NULL
        ORDER BY s.plays DESC
    """
    missing_albums = pd.read_sql(query, conn)

print("Songs with no album data:")
print(missing_albums)

In [ ]:
query = """
SELECT title, artist, plays
FROM songs
WHERE plays > (SELECT AVG(plays) FROM songs)
ORDER BY plays DESC
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print("Songs above average plays:")
print(df)

In [ ]:
query = """
SELECT artist, avg_plays
FROM (
    SELECT artist, 
           AVG(plays) AS avg_plays
    FROM songs
    GROUP BY artist
) AS artist_averages
WHERE avg_plays > 1000
ORDER BY avg_plays DESC
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print("Row count:", len(df))
print()
print(df)

In [ ]:
query = """
SELECT s.title,
       s.artist,
       s.plays
FROM songs s
WHERE EXISTS (
    SELECT 1
    FROM artists ar
    WHERE ar.name = s.artist
    AND ar.genre IS NOT NULL
)
ORDER BY s.plays DESC
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print("Row count:", len(df))
print(df)

In [ ]:
query = """
SELECT
    title,
    year,
    CASE
        WHEN year < 1980 THEN 'Classic'
        WHEN year BETWEEN 1980 AND 1999 THEN 'Modern'
        WHEN year >= 2000 THEN 'Contemporary'
        ELSE 'Unknown'
    END AS era_label
FROM songs
ORDER BY year ASC
"""

with engine.connect() as conn:
    df_era = pd.read_sql(query, conn)

print("Songs Eras:")
print(df_era)

In [ ]:
query = """
    SELECT
    ar.name AS artist_name,
    ar.genre,
    -- 1. Subquery: Calculate total plays for THIS specific artist
    (SELECT SUM(plays) FROM songs s WHERE s.artist = ar.name) AS total_plays,

    -- 2. EXISTS: Check if this artist has a record in the albums table
    EXISTS (SELECT 1 FROM albums al WHERE al.artist_id = ar.id) AS has_album_record,

    -- 3. CASE: Create a success tier based on the subquery result
    CASE
        WHEN (SELECT SUM(plays) FROM songs s WHERE s.artist = ar.name) > 1500 THEN 'Superstar'
        WHEN (SELECT SUM(plays) FROM songs s WHERE s.artist = ar.name) BETWEEN 1000 AND 1500 THEN 'Hit Maker'
        ELSE 'Rising Star'
    END AS career_status
FROM artists ar
ORDER BY total_plays DESC
"""

with engine.connect() as conn:
    complex_df = pd.read_sql(query,conn)

print("Artist Maker Analysis:")
print(complex_df)

In [ ]:
query = """
SELECT 
    al.title AS album_name,
    -- If the artist name is missing for some reason, show 'Unknown Artist'
    COALESCE(ar.name, 'Unknown Artist') AS artist_name,
    al.total_tracks,
    -- Window Function: Get the average tracks per album for THAT artist
    AVG(al.total_tracks) OVER(PARTITION BY al.artist_id) AS artist_avg_tracks,
    -- CASE: Compare this album to the artist's average
    CASE 
        WHEN al.total_tracks > AVG(al.total_tracks) OVER(PARTITION BY al.artist_id) THEN 'Above Average'
        WHEN al.total_tracks < AVG(al.total_tracks) OVER(PARTITION BY al.artist_id) THEN 'Below Average'
        ELSE 'On Par'
    END AS track_count_status
FROM albums al
LEFT JOIN artists ar ON al.artist_id = ar.id
WHERE EXISTS (
    -- Only show albums for artists that actually have songs in our 'songs' table
    SELECT 1 FROM songs s WHERE s.artist = ar.name
)
"""

with engine.connect() as conn:
    pro_df = pd.read_sql(query, conn)

print("Advanced Album Analysis:")
print(pro_df)

In [ ]:
# Query uses COALESCE to handle missing data and looks for specific 'tags'
query = """
SELECT
    title,
    artist,
    -- If 'genre' is NULL, we show 'Miscellaneous'
    COALESCE( genre, 'Miscellaneous') AS genre_cleaned,
    -- RANK() is a window function that orders songs by plays within their genre 
    RANK() OVER(PARTITION BY genre ORDER BY plays DESC) as rank_in_genre
FROM songs s
LEFT JOIN artists ar ON s.artist = ar.name
"""

with engine.connect() as conn:
    df_pro = pd.read_sql(query, conn)
print(df_pro)

In [ ]:
# Reusable function that takes a 'minimum plays' number as an input. 
# It pulls that data from SQL, uses Python to tag 'viral' songs, 
# and automatically saves a clean CSV file

def export_top_songs(min_plays, file_name):
    # 1. Get the data using SQL
    query = f"SELECT * FROM songs WHERE plays >= {min_plays}"

    with engine.connect() as conn:
        df = pd.read_sql(query, conn)

    # 2. Add a Python-calculated column
    df['is_viral'] = df['plays'] > 1500

    # 3. Save it to a CSV 
    df.to_csv(file_name, index=False)
    print(f" Exported {len(df)} songs to {file_name}")

export_top_songs(1000, "popular_songs_report.csv")   

In [ ]:
from faker import Faker
import random

fake = Faker()

def generate_fake_songs(number_of_songs):
    new_songs = []

    # We use a loop to create each song one by one
    for _ in range(number_of_songs):
        song_entry = {
            "title": fake.sentence(nb_words=3).replace(".", "").title(),   # Generates a sentence with about 3 words
            "artist": fake.name(),                                         # Generates a fake name
            "year": random.randint(1960, 2024),                            # Random year
            "plays": random.randint(100, 3000)                             # Random play count
        }
        new_songs.append(song_entry)

    # Convert our list of dictionaries to Pandas DF
    # List - > Table
    fake_df = pd.DataFrame(new_songs)
    return fake_df

    # Push to PostgreSQL
    # 'append' adds to the end of the table, not touching what's there
def load_to_postgres(fake_df, mode="append"):
    with engine.connect() as conn:
        fake_df.to_sql('songs', conn, if_exists=mode, index=False)
    count = len(fake_df)
    print(f" Successfully added {count} fake songs to the database!")

df_to_check = generate_fake_songs(50)
#print(df_to_check.head())
#load_to_postgres(df_to_check, mode="append")

In [ ]:
# The "Top Artist" Report
query = """
WITH artist_stats AS (
    -- This 'temporary table' counts songs and averages plays
    SELECT 
        artist, 
        COUNT(*) as song_count, 
        AVG(plays) as avg_plays
    FROM songs
    GROUP BY artist
)
SELECT * FROM artist_stats 
WHERE song_count = 1
ORDER BY avg_plays DESC
"""

with engine.connect() as conn:
    df_stats = pd.read_sql(query, conn)

print("Artists with multiple hits:")
print(df_stats)

In [ ]:
from sqlalchemy import text
# Find songs with very few plays and delete them to keep the DB 'Premium'
threshold = 200

delete_query = text(f"DELETE FROM songs WHERE plays < {threshold}")

with engine.connect() as conn:
    # We use 'execute' instead of 'read_sql' because we aren't getting data back
    result = conn.execute(delete_query)
    print(f"🧹 Cleaned up {result.rowcount} low-play songs.")

In [ ]:
from faker import Faker
from sqlalchemy import text
import random

fake = Faker()

# 1. Define a small list of artists so they repeat
top_artists = ["Queen", "Nirvana", "Adele", "The Weeknd", "Unknown"]

def generate_complex_data(number_of_songs):
    new_songs = []
    for _ in range(number_of_songs):
        # Use random.choice to pick from list above
        artist_name = random.choice(top_artists)
        
        # Make some data NULL
        year = random.choice([1995, 2010, 2022, None]) 
        
        new_songs.append({
            "title": fake.sentence(nb_words=3).replace(".", "").title(),
            "artist": artist_name,
            "year": year,
            "plays": random.randint(50, 2500)
        })
    return pd.DataFrame(new_songs)

# Push to PostgreSQL
# 'append' adds to the end of the table, not touching what's there
def load_to_postgres(fake_df, mode="append"):
    with engine.connect() as conn:
        fake_df.to_sql('songs', conn, if_exists=mode, index=False)
    count = len(fake_df)
    print(f" Successfully added {count} fake songs to the database!")

# Run it
messy_df = generate_complex_data(50)
load_to_postgres(messy_df, mode="append")

In [ ]:
from sqlalchemy import text

query = """
-- CTE 1: Calculate stats for everyone
WITH base_stats AS (
    SELECT 
        artist, 
        COUNT(*) as song_count, 
        AVG(plays) as avg_plays
    FROM songs
    GROUP BY artist
),
-- CTE 2: Filter "Legends"
legends AS (
    SELECT *, 'Legend (Multi-Song)' as category 
    FROM base_stats WHERE song_count > 1
),
-- CTE 3: Filter "One-Hits"
one_hits AS (
    SELECT *, 'One-Hit Wonder' as category 
    FROM base_stats WHERE song_count = 1
)
-- CTE 4: Combine to compare
SELECT * FROM legends
UNION ALL
SELECT * FROM one_hits
ORDER BY category, avg_plays DESC
"""

with engine.connect() as conn:
    # Use text(query)
    comparison_df = pd.read_sql(text(query), conn)

print("Comparison Report:")
print(comparison_df)

In [ ]:
from sqlalchemy import text

query = """
SELECT
    title, 
    artist,
    -- If year is NULL, show 0. If artist is 'Unknown', ahow 'Independent'
    COALESCE(year, 0) as year_cleaned,
    CASE 
        WHEN artist = 'Unknown' THEN 'Independent'
        ELSE artist
    END as artist_category
FROM songs
WHERE year IS NULL OR artist = 'Unknown'    
"""

with engine.connect() as conn:
    df_nulls = pd.read_sql(query, conn)

print("Rows to clean:")
print(df_nulls.head())

In [ ]:
query = """
SELECT
    title,
    UPPER(title) as shout_title,
    LOWER(artist) as quiet_artist,
    TRIM(title) as trimmed_title
FROM songs
LIMIT 10          
"""

with engine.connect() as conn:
    df_strings = pd.read_sql(query,conn)
print("Strings Manipulation Result:")
print(df_strings.head())

In [ ]:
query = """
SELECT * FROM songs
WHERE year > 2025
    OR plays <= 0
    OR LENGTH(title) < 2
"""
with engine.connect() as conn:
    df_errors = pd.read_sql(query, conn)

if df_errors.empty:
    print("Logic Check Passed: No faulty data")
else:
    print("Logic Check Failed: Found faulty rows!")
    print(df_errors)